# 08 Live HF spectator

Run a **multi-agent simulation** with **per-agent Hub `model_id`** values. Choose **how** models are picked and **where** inference runs:

- **HF Inference API** — needs `HF_TOKEN` (`.env` or `hf_token.txt`); uses the curated **open-access** pool from `agent_rpg.model_catalog` (no Meta Llama gating).
- **Local Transformers** — runs on this machine (`pip install -e '.[local]'`); first run **downloads** each selected checkpoint from the Hub (open models; token optional but helps with rate limits).

**Assignment:** either a **multi-select pool** plus rotate/shuffle/random, or **one dropdown per agent** so each speaker can use a different model explicitly.

**World & cast:** pick a **premise**, optional **environment** text overrides, shared **character** goal/archetype/occupation, and **turn order / memory / stop phrase** — all in section 2 before you build the scenario.

> **Import issues after `git pull`?** Kernel → Restart, then run from the top.

> **HTTP 400 `model_not_supported`?** On **HF Inference API**, pick **Qwen2.5 7B** (or another **HF router** model) in §2 — not 1.5B/3B. Or switch to **Local Transformers**.

> **HTTP 402?** HF Inference billing — see [billing](https://huggingface.co/settings/billing) or switch to **Local Transformers**.

> **Controls not rendering?** Use the **Models / World / Characters / Run** tabs in section 2; re-run that cell after a kernel restart.


In [1]:
from pathlib import Path
import sys
for base in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    src = base / "src"
    if (src / "agent_rpg").is_dir():
        if str(src) not in sys.path:
            sys.path.insert(0, str(src))
        break
from agent_rpg.repo_root import find_repo_root
ROOT = find_repo_root()
print("Repository root:", ROOT)


Repository root: /Users/morningstar/Desktop/Cold_Storage/Agent_RPG_Simulator


## 1 — Environment

In [2]:
import os

try:
    from dotenv import load_dotenv

    load_dotenv(ROOT / ".env")
except ImportError:
    pass

if not os.environ.get("HF_TOKEN"):
    for _name in ("hf_token.txt", "HF_TOKEN.txt"):
        _p = ROOT / _name
        if _p.is_file():
            _line = (_p.read_text(encoding="utf-8").strip().splitlines() or [""])[0].strip()
            if _line:
                os.environ["HF_TOKEN"] = _line
            break

if os.environ.get("HF_TOKEN"):
    print("HF_TOKEN is set (value hidden).")
else:
    print("HF_TOKEN not set — required for HF Inference API; optional for local downloads (rate limits).")

HF_TOKEN is set (value hidden).


## 2 — Controls (ipywidgets)

Four **tabs** — **Models**, **World**, **Characters**, **Run**. Re-run this cell after **Kernel → Restart** if you only see `VBox(...)` text instead of buttons and sliders.

- **Models:** HF cloud vs local `transformers`, pool or per-agent assignment, device.
- **World:** premise and optional environment overrides.
- **Characters:** shared goal / archetype / occupation.
- **Run:** agents, rounds, seed, turn order, memory, stop phrase, thought phase.

> **Controls blank in Cursor / VS Code?** Select the `.venv` kernel, run **§1** then this cell. If still broken: `pip install -r requirements-notebooks.txt` and reload the window.


In [ ]:
from agent_rpg.live_spectator_ui import build_and_display_live_spectator

ui = build_and_display_live_spectator()

# Names expected by sections 3–5 below
exec_mode = ui.exec_mode
assign_mode = ui.assign_mode
pool_sel = ui.pool_sel
strat = ui.strat
agent_dds = ui.agent_dds
n_agents = ui.n_agents
rounds = ui.rounds
seed = ui.seed
stream_tokens = ui.stream_tokens
thoughts = ui.thoughts
device_map = ui.device_map
world_premise_dd = ui.world_premise_dd
env_setting_ta = ui.env_setting_ta
env_backstory_ta = ui.env_backstory_ta
env_notes_ta = ui.env_notes_ta
char_goal_dd = ui.char_goal_dd
char_arch_dd = ui.char_arch_dd
char_occ_dd = ui.char_occ_dd
turn_order_dd = ui.turn_order_dd
memory_turns_sld = ui.memory_turns_sld
stop_phrase_txt = ui.stop_phrase_txt
labels = ui.labels
label_to_id = ui.label_to_id



## 3 — Build scenario and wire models / backends

In [ ]:
from agent_rpg import SimulationEngine, build_random_scenario
from agent_rpg.backends.hf_inference import HuggingFaceInferenceBackend
from agent_rpg.model_catalog import default_model_id_for_execution, warn_if_not_hf_router
from agent_rpg.multi_model import assign_models_to_agents, set_router_model_if_reactive

_world_title = (world_premise_dd.value or "").strip() or None
scenario = build_random_scenario(
    seed=int(seed.value),
    num_agents=int(n_agents.value),
    max_rounds=int(rounds.value),
    model_id=default_model_id_for_execution(exec_mode.value),
    world_title=_world_title,
    turn_order=turn_order_dd.value,
    memory_turns=int(memory_turns_sld.value),
    stop_phrase=(stop_phrase_txt.value or "").strip(),
    enable_thought_phase=bool(thoughts.value),
)

if assign_mode.value == "Pick per agent":
    for i, ag in enumerate(scenario.agents):
        ag.model_id = label_to_id[agent_dds[i].value]
else:
    chosen_labels = tuple(pool_sel.value) if pool_sel.value else (labels[0],)
    model_pool = [label_to_id[l] for l in chosen_labels]
    assign_models_to_agents(
        scenario,
        model_pool,
        strategy=strat.value,
        rng=__import__("random").Random(int(seed.value)),
    )

set_router_model_if_reactive(scenario, scenario.agents[0].model_id)

_bad_hf = warn_if_not_hf_router([a.model_id for a in scenario.agents], execution_mode=exec_mode.value)
if _bad_hf:
    print("Warning: not on HF Inference router (use Qwen 7B or enable another provider):", _bad_hf)

_es = env_setting_ta.value.strip()
if _es:
    scenario.world.setting = _es
_eb = env_backstory_ta.value.strip()
if _eb:
    scenario.world.backstory = _eb
_en = env_notes_ta.value.strip()
if _en:
    scenario.world.worldbuilding_notes = _en

_g = (char_goal_dd.value or "").strip()
_a = (char_arch_dd.value or "").strip()
_o = (char_occ_dd.value or "").strip()
for ag in scenario.agents:
    if _g:
        pv = dict(ag.prompt_variables)
        pv["goal"] = _g
        ag.prompt_variables = pv
    if _a:
        ag.archetype = _a
    if _o:
        ag.occupation = _o

if exec_mode.value == "Local Transformers":
    for ag in scenario.agents:
        ag.backend = "transformers_local"
else:
    for ag in scenario.agents:
        ag.backend = "auto"

for ag in scenario.agents:
    ag.max_new_tokens = min(ag.max_new_tokens, 320)
    ag.temperature = min(ag.temperature, 0.85)

print("World:", scenario.world.title)
print("Turn order:", scenario.orchestration.turn_order, "| memory turns:", scenario.orchestration.memory_turns)
if scenario.orchestration.stop_phrase:
    print("Stop phrase:", repr(scenario.orchestration.stop_phrase))
print("Execution:", exec_mode.value)
for a in scenario.agents:
    print(f"  {a.display_name} ({a.occupation}): {a.model_id}  backend={a.backend}")


World: Storm over the trade docks
Execution: Local Transformers
  Mara Vale (notary): Qwen/Qwen2.5-1.5B-Instruct  backend=transformers_local
  Ilo Ironhand (smuggler): Qwen/Qwen2.5-1.5B-Instruct  backend=transformers_local
  Silas Reed (guard): Qwen/Qwen2.5-1.5B-Instruct  backend=transformers_local


## 4 — Spectate the run
Local runs ignore **Stream tokens** (no streaming path in `TransformersLocalBackend` yet).

In [5]:
from IPython.display import Markdown, display

_agent_by_id = {a.agent_id: a for a in scenario.agents}


def _agent_line(aid: str | None) -> str:
    if not aid:
        return "?"
    ag = _agent_by_id.get(aid)
    if ag is None:
        return aid
    occ = (ag.occupation or "").strip()
    arch = (ag.archetype or "").strip()
    if occ and arch:
        return f"{ag.display_name} ({occ} — {arch})"
    if occ:
        return f"{ag.display_name} ({occ})"
    if arch:
        return f"{ag.display_name} ({arch})"
    return ag.display_name


def on_event(ev):
    if ev.event_type == "world_event":
        eid = ev.payload.get("event_id", "")
        desc = ev.payload.get("description", "")
        sub = f"`{eid}` — " if eid else ""
        display(Markdown(f"**World** (round {ev.round}): {sub}{desc}"))
    elif ev.event_type == "thought":
        who = _agent_line(ev.agent_id)
        display(Markdown(f"_Thought — **{who}**_: {ev.payload.get('text', '')}"))
    elif ev.event_type == "message":
        who = _agent_line(ev.agent_id)
        say = ev.payload.get("text", "")
        display(Markdown(f"**{who}**: {say}"))
    elif ev.event_type == "router":
        display(Markdown(f"_Router_: next speaker hint `{ev.payload.get('chosen')!s}`"))
    elif ev.event_type == "error":
        display(Markdown(f"⚠ **error**: `{ev.payload}`"))
    elif ev.event_type == "system":
        display(Markdown(f"_System_: `{ev.payload}`"))

local_backend = None
_use_local = exec_mode.value == "Local Transformers"
if _use_local:
    from agent_rpg.backends.transformers_local import TransformersLocalBackend

    _dm = device_map.value.strip() or "auto"
    local_backend = TransformersLocalBackend(device_map=_dm)

backend = HuggingFaceInferenceBackend()
llm_extras: dict = {}
if stream_tokens.value and not _use_local:

    def chunk_cb(t: str) -> None:
        print(t, end="", flush=True)

    llm_extras = {"stream": True, "chunk_callback": chunk_cb}

run_id = f"live_spec_{int(seed.value)}"
print("--- simulation start ---\n")
out = SimulationEngine(scenario).run(
    backend,
    output_dir=ROOT / "runs",
    run_id=run_id,
    seed=int(seed.value),
    on_event=on_event,
    llm_extras=llm_extras,
    local_backend=local_backend,
)
print("\n--- done ---\nJSONL:", out / "events.jsonl")

--- simulation start ---



_System_: `{'message': 'simulation_start', 'scenario_id': 'rand_7c2c4cfd15'}`

**World** (round 0): `evt_0_7227` — A tugboat's steam whistle slices through the argument; a foreman shouts that the manifest copy in the harbormaster's office no longer matches the one pinned inside this warehouse.

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the disk.
[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_i

⚠ **error**: `{'stage': 'parse', 'error': 'empty_say', 'raw': '```json\n{\n  "thought": "",\n  "say": "",\n  "directed_at": null\n}\n```'}`

**Ilo Ironhand (smuggler — impatient merchant)**: 

[transformers] Both `max_new_tokens` (=320) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


KeyboardInterrupt: 

## 5 — Optional: quick report

In [6]:
from agent_rpg import ReportBuilder

# Same directory as section 4 (`run_id` + `ROOT / "runs"`); avoids NameError if the run cell stopped early or was skipped.
_events_jsonl = ROOT / "runs" / f"live_spec_{int(seed.value)}" / "events.jsonl"
rb = ReportBuilder.from_jsonl(_events_jsonl)
d = rb.to_dict(scenario=scenario)
print("Messages:", d["summary"]["message_count"], "Errors:", d["summary"]["error_count"])
print("Gini:", round(d["social_dynamics"]["gini_turns"], 4))

NameError: name 'out' is not defined